# EDA-Streaming

### All Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import seaborn as sns

### Load Data

In [ ]:
df = pd.read_csv("spreadsheets/streaming.csv")
df.groupby("streaming").count()
df.info()

### EDA

#### Missing Values - Search

In [ ]:
streaming_na = df.isna().groupby(df["streaming"]).sum()

In [ ]:
plt.figure(figsize=(14, 7))
streaming_na.T.plot(
    kind='bar', 
    figsize=(14, 7), 
    title='Count of Missing Values by Variable, Compared by Streaming Service'
)

plt.ylabel('Number of Missing Values')
plt.xlabel('Dataset Variables')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Streaming Service')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#### Anomaly - Search

In [ ]:
df["country"].info() # Clean and Map

In [ ]:
df["release_year"].describe()

In [ ]:
df["rating"].unique() # Clean and Map

In [ ]:
df["duration"].unique() # Clean and discretise

In [ ]:
df["listed_in"].unique() # Clean and Map

In [ ]:
df["streaming"].unique()

#### Clean

##### Duration TV Shows - Seasons

In [ ]:
print("TV Show (Before):")
movie_durations_before = df[df["type"] == "TV Show"]["duration"]
print(f"Unique values (array): {movie_durations_before.unique()}")

count_before = movie_durations_before.nunique()
print(f"Number of unique values (Before): {count_before}")

df.loc[df["type"] == "TV Show", "duration"] = (
    df[df["type"] == "TV Show"]["duration"]
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype(float)
    .astype('Int64')
)


print("\nAfter conversion:")
movie_durations_after = df[df["type"] == "TV Show"]["duration"]
print(f"Unique values (array): {movie_durations_after.unique()}")

count_after = movie_durations_after.nunique()
print(f"Number of unique values (After): {count_after}")

print("\nVerification:")
print(f"Is the count the same? {count_before == count_after}")

##### Duration Movies - Minutes

In [ ]:
print("Movies (Before):")
movie_durations_before = df[df["type"] == "Movie"]["duration"]
print(f"Unique values (array): {movie_durations_before.unique()}")

count_before = movie_durations_before.nunique()
print(f"Number of unique values (Before): {count_before}")

df.loc[df["type"] == "Movie", "duration"] = (
    df[df["type"] == "Movie"]["duration"]
    .astype(str)
    .str.extract(r'(\d+)', expand=False)
    .astype(float)
    .astype('Int64')
)


print("\nAfter conversion:")
movie_durations_after = df[df["type"] == "Movie"]["duration"]
print(f"Unique values (array): {movie_durations_after.unique()}")

count_after = movie_durations_after.nunique()
print(f"Number of unique values (After): {count_after}")

print("\nVerification:")
print(f"Is the count the same? {count_before == count_after}")

##### Rating

In [ ]:
print("Rating (Before):")
rating_before = df["rating"]
print(f"Unique values (array): {rating_before.unique()}")

count_before = rating_before.nunique()
print(f"Number of unique values (Before): {count_before}")

rating_mapping = {
    'G': 'G',
    'PG': 'PG',
    'PG-13': 'PG-13',
    'R': 'R',
    'NC-17': 'NC-17',

    'TV-Y': 'TV-Y',
    'TV-Y7': 'TV-Y7',
    'TV-Y7-FV': 'TV-Y7',
    'TV-G': 'TV-G',
    'TV-PG': 'TV-PG',
    'TV-14': 'TV-14',
    'TV-MA': 'TV-MA',

    'TV-NR': 'UNRATED',
    'NR': 'UNRATED',
    'UR': 'UNRATED',
    'UNRATED': 'UNRATED',
    'NOT RATED': 'UNRATED',
    'NOT_RATE': 'UNRATED',
    
    '13+': 'TV-14',
    '16+': 'TV-MA',
    '18+': 'TV-MA',
    '7+': 'TV-Y7',
    '16': 'TV-MA',
    'AGES_16_': 'TV-MA',
    'AGES_18_': 'TV-MA',
    'ALL_AGES': 'G',
    'ALL': 'G'
}

df["rating_clean"] = df["rating"].apply(
    lambda x: (
        'UNRATED' 
        if pd.isna(x) 
        or any(term in str(x).lower() for term in ['min', 'season'])
        else rating_mapping.get(str(x).strip(), 'UNRATED')
    )
)

print("\nAfter conversion:")
rating_after = df["rating_clean"]
print(f"Unique values (array): {rating_after.unique()}")

count_after = rating_after.nunique()
print(f"Number of unique values (After): {count_after}")

print("\nVerification:")
print(f"Is the count the same? {count_before == count_after}")
df["rating"] = df["rating_clean"]


##### Listed in

In [ ]:
df['genre'] = df['listed_in'].apply(
    lambda x: [genre.strip().lower() for genre in x.replace('&', ',').replace('/', ',').replace(' and ', ',').split(',')]
)

##### Country

In [ ]:
df['country'] = df['country'].apply(
    lambda x: [country.strip().lower() for country in x.split(',')] if isinstance(x, str) else x
)

#### Discretise

##### Pre-discretise

In [ ]:
df['duration_class'] = pd.Series(np.nan, index=df.index, dtype='object')

##### Duration TV Shows - Seasons to Class

|   Class  | Num Seasons |
|----------|-------------|
|Miniseries|    N <= 1   |
|Short     | 2 <= N <= 3 |
|Medium    | 4 <= N <= 7 |
|Long      |    N >= 8   |

In [ ]:
bins = [0, 1, 3, 7, np.inf]
labels = ['Miniseries', 'Short', 'Medium', 'Long']

tv_show_mask = df['type'] == 'TV Show'

df.loc[tv_show_mask, 'duration_class'] = pd.cut(
    df.loc[tv_show_mask, 'duration'],
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

print(df[tv_show_mask]['duration_class'].value_counts(dropna=False))

##### Duration Movies - Minutes to Class

|   Class  | Time in Minutes |
|----------|-----------------|
|   Short  |     T <= 40     |
| Standard |  40 < T <= 100  |
|  Feature | 100 < T <= 180  |
|   Long   |     T > 180     |

In [ ]:
bins = [0, 40, 100, 180, np.inf]
labels = ['Short', 'Standard', 'Feature', 'Long']

movie_mask = df['type'] == 'Movie'
movie_durations = df.loc[movie_mask, 'duration'].dropna()
df.loc[movie_durations.index, 'duration_class'] = pd.cut(
    movie_durations,
    bins=bins,
    labels=labels,
    right=True,
    include_lowest=True
)

print(df[movie_mask]['duration_class'].value_counts(dropna=False))

#### Map

##### Listed in

In [ ]:
genre_mapping = {
    # Action & Adventure
    'action': 'Action-adventure',
    'action-adventure': 'Action-adventure',
    'adventure': 'Action-adventure',
    'disaster': 'Action-adventure',
    'military': 'Action-adventure',
    'superhero': 'Action-adventure',
    'survival': 'Action-adventure',
    'tv action': 'Action-adventure',
    'war': 'Action-adventure',
    'western': 'Action-adventure',

    # Adult animation
    'adult animation': 'Adult animation',

    # Animation
    'animation': 'Animation',
    'cartoons': 'Animation',

    # Anime
    'anime': 'Anime',
    'anime features': 'Anime',
    'anime series': 'Anime',

    # Classic & Cult
    'classic': 'Classic-cult',
    'classic movies': 'Classic-cult',
    'classics': 'Classic-cult',
    'cult movies': 'Classic-cult',
    'cult tv': 'Classic-cult',

    # Comedy
    'buddy': 'Comedy',
    'comedy': 'Comedy',
    'comedies': 'Comedy',
    'parody': 'Comedy',
    'romantic comedy': 'Comedy',
    'sitcom': 'Comedy',
    'sketch comedy': 'Comedy',
    'stand up': 'Comedy',
    'stand-up comedy': 'Comedy',
    'tv comedies': 'Comedy',

    # Crime
    'cop': 'Crime',
    'crime': 'Crime',
    'crime tv shows': 'Crime',
    'police': 'Crime',

    # Documentary
    'biographical': 'Documentary',
    'documentaries': 'Documentary',
    'documentary': 'Documentary',
    'docuseries': 'Documentary',
    'historical': 'Documentary',
    'history': 'Documentary',

    # Drama
    'drama': 'Drama',
    'dramas': 'Drama',
    'melodrama': 'Drama',
    'soap opera': 'Drama',
    'tv dramas': 'Drama',

    # Family
    'children': 'Family',
    'family': 'Family',
    'family movies': 'Family',
    'kids': 'Family',
    "kids' tv": 'Family',
    
    # Indie
    'arthouse': 'Indie',
    'independent movies': 'Indie',

    # LGBTQIA+
    'lgbtq': 'Lgbtqia',
    'lgbtq movies': 'Lgbtqia',
    'lgbtq+': 'Lgbtqia',

    # Mystery
    'mystery': 'Mystery',
    'tv mysteries': 'Mystery',

    # Music & Musical
    'concert film': 'Music-musical',
    'concerts': 'Music-musical',
    'dance': 'Music-musical',
    'music': 'Music-musical',
    'music videos': 'Music-musical',
    'musical': 'Music-musical',
    'musicals': 'Music-musical',

    # Science & Nature
    'animals': 'Science-nature',
    'nature': 'Science-nature',
    'nature tv': 'Science-nature',
    'science': 'Science-nature',
    'technology': 'Science-nature',
    
    # Special interest
    'arts': 'Special interest',
    'culture': 'Special interest',
    'entertainment': 'Special interest',
    'medical': 'Special interest',
    'special interest': 'Special interest',
    
    # News
    'news': 'News',

    # Reality & Lifestyle
    'competition': 'Reality-lifestyle',
    'cooking': 'Reality-lifestyle',
    'fitness': 'Reality-lifestyle',
    'food': 'Reality-lifestyle',
    'game show': 'Reality-lifestyle',
    'game shows': 'Reality-lifestyle',
    'health': 'Reality-lifestyle',
    'lifestyle': 'Reality-lifestyle',
    'reality': 'Reality-lifestyle',
    'reality tv': 'Reality-lifestyle',
    'travel': 'Reality-lifestyle',
    'unscripted': 'Reality-lifestyle',
    'wellness': 'Reality-lifestyle',

    # Talk Show
    'late night': 'Talk show',
    'talk show': 'Talk show',
    'talk shows': 'Talk show',

    # Religion
    'faith': 'Religion',
    'spirituality': 'Religion',

    # Romance
    'romance': 'Romance',
    'romantic movies': 'Romance',
    'romantic tv shows': 'Romance',

    # Sci-Fi
    'sci-fi': 'Science fiction',
    'science fiction': 'Science fiction',
    'tv sci-fi': 'Science fiction',

    # Fantasy
    'fantasy': 'Fantasy',

    # Sports
    'sports': 'Sports',
    'sports movies': 'Sports',

    # Teen
    'coming of age': 'Teen',
    'teen': 'Teen',
    'teen tv shows': 'Teen',
    'young adult audience': 'Teen',

    # Thriller
    'black stories': 'Thriller',
    'espionage': 'Thriller',
    'spy': 'Thriller',
    'suspense': 'Thriller',
    'thriller': 'Thriller',
    'thrillers': 'Thriller',
    'tv thrillers': 'Thriller',
    
    # Horror
    'horror': 'Horror',
    'horror movies': 'Horror',
    'tv horror': 'Horror',

    # Variety
    'variety': 'Variety',

    # Remove
    '': 'Remove',
    'movies': 'Remove',
    'series': 'Remove',
    'tv shows': 'Remove',
    'british tv shows': 'Remove',
    'international': 'Remove',
    'international movies': 'Remove',
    'international tv shows': 'Remove',
    'korean tv shows': 'Remove',
    'latino': 'Remove',
    'spanish-language tv shows': 'Remove',
    'anthology': 'Remove',
}
df_exploded = df.explode("genre")
df_exploded['genre'] = df_exploded['genre'].replace(genre_mapping)
df_exploded = df_exploded[df_exploded['genre'] != 'Remove']

print(df_exploded["genre"].unique())

In [ ]:
cleaned_genres_series = df_exploded.groupby(df_exploded.index)['genre'].apply(
    lambda x: x.unique().tolist()
)
df["genre"] = cleaned_genres_series

##### Country

In [ ]:
country_mapping = {
    # Mains Countries
    'united states': 'United States',
    'united kingdom': 'United Kingdom',
    'india': 'India',
    'japan': 'Japan',
    'south korea': 'South Korea',
    'china': 'China',
    'france': 'France',
    'germany': 'Germany',
    'canada': 'Canada',
    'australia': 'Australia',
    'spain': 'Spain',
    'mexico': 'Mexico',
    'brazil': 'Brazil',
    'nigeria': 'Nigeria',
    'turkey': 'Turkey',
    'hong kong': 'Hong kong',
    'taiwan': 'Taiwan',
    'italy': 'Italy',
    'russia': 'Russia',
    'argentina': 'Argentina',
    'south africa': 'South Africa',

    # Special Cases
    'west germany': 'Germany',
    'east germany': 'Germany',
    'soviet union': 'Russia',
    'vatican city': 'Vatican',
    '': 'Unknown',
    np.nan: 'Unknown',

    # Latin America & Caribbean
    'venezuela': 'Latin America - Caribbean',
    'colombia': 'Latin America - Caribbean',
    'uruguay': 'Latin America - Caribbean',
    'chile': 'Latin America - Caribbean',
    'peru': 'Latin America - Caribbean',
    'guatemala': 'Latin America - Caribbean',
    'paraguay': 'Latin America - Caribbean',
    'ecuador': 'Latin America - Caribbean',
    'cuba': 'Latin America - Caribbean',
    'nicaragua': 'Latin America - Caribbean',
    'dominican republic': 'Latin America - Caribbean',
    'panama': 'Latin America - Caribbean',
    'costa rica': 'Latin America - Caribbean',
    'puerto rico': 'Latin America - Caribbean',
    'cayman islands': 'Latin America - Caribbean',
    'bermuda': 'Latin America - Caribbean',
    'bahamas': 'Latin America - Caribbean',
    'jamaica': 'Latin America - Caribbean',
    
    # Europe
    'ireland': 'Europe Other',
    'belgium': 'Europe Other',
    'romania': 'Europe Other',
    'switzerland': 'Europe Other',
    'bulgaria': 'Europe Other',
    'poland': 'Europe Other',
    'denmark': 'Europe Other',
    'netherlands': 'Europe Other',
    'hungary': 'Europe Other',
    'sweden': 'Europe Other',
    'iceland': 'Europe Other',
    'norway': 'Europe Other',
    'austria': 'Europe Other',
    'luxembourg': 'Europe Other',
    'portugal': 'Europe Other',
    'serbia': 'Europe Other',
    'malta': 'Europe Other',
    'belarus': 'Europe Other',
    'cyprus': 'Europe Other',
    'croatia': 'Europe Other',
    'albania': 'Europe Other',
    'georgia': 'Europe Other',
    'slovakia': 'Europe Other',
    'ukraine': 'Europe Other',
    'armenia': 'Europe Other',
    'latvia': 'Europe Other',
    'liechtenstein': 'Europe Other',
    'slovenia': 'Europe Other',
    'lithuania': 'Europe Other',
    'montenegro': 'Europe Other',
    'monaco': 'Europe Other',
    'kosovo': 'Europe Other',
    'czech republic': 'Europe Other',
    'greece': 'Europe Other',
    'finland': 'Europe Other',

    # East Southeast Asia
    'singapore': 'East Southeast Asia',
    'thailand': 'East Southeast Asia',
    'indonesia': 'East Southeast Asia',
    'malaysia': 'East Southeast Asia',
    'vietnam': 'East Southeast Asia',
    'philippines': 'East Southeast Asia',
    'cambodia': 'East Southeast Asia',
    'mongolia': 'East Southeast Asia',

    # Asia
    'nepal': 'South Central Asia',
    'bangladesh': 'South Central Asia',
    'pakistan': 'South Central Asia',
    'sri lanka': 'South Central Asia',
    'kazakhstan': 'South Central Asia',
    'afghanistan': 'South Central Asia',
    'azerbaijan': 'South Central Asia',

    # Middle East North Africa
    'jordan': 'Middle East North Africa',
    'israel': 'Middle East North Africa',
    'algeria': 'Middle East North Africa',
    'saudi arabia': 'Middle East North Africa',
    'egypt': 'Middle East North Africa',
    'kuwait': 'Middle East North Africa',
    'lebanon': 'Middle East North Africa',
    'syria': 'Middle East North Africa',
    'united arab emirates': 'Middle East North Africa',
    'qatar': 'Middle East North Africa',
    'palestine': 'Middle East North Africa',
    'iraq': 'Middle East North Africa',
    'iran': 'Middle East North Africa',
    'morocco': 'Middle East North Africa',
    'tunisia': 'Middle East North Africa',
    
    # Africa Subsaharan
    'ghana': 'Africa Subsaharan',
    'burkina faso': 'Africa Subsaharan',
    'ethiopia': 'Africa Subsaharan',
    'cameroon': 'Africa Subsaharan',
    'kenya': 'Africa Subsaharan',
    'senegal': 'Africa Subsaharan',
    'namibia': 'Africa Subsaharan',
    'angola': 'Africa Subsaharan',
    'mozambique': 'Africa Subsaharan',
    'zimbabwe': 'Africa Subsaharan',
    'malawi': 'Africa Subsaharan',
    'botswana': 'Africa Subsaharan',
    'somalia': 'Africa Subsaharan',
    'sudan': 'Africa Subsaharan',
    'uganda': 'Africa Subsaharan',
    'tanzania': 'Africa Subsaharan',
    'mauritius': 'Africa Subsaharan',

    # Oceania
    'new zealand': 'Oceania Other',
    'samoa': 'Oceania Other',
    'french polynesia': 'Oceania Other',
}

df_exploded = df.explode("country")
df_exploded["country"] = df_exploded["country"].replace(country_mapping)
print(df_exploded["country"].unique())

In [ ]:
cleaned_country_series = df_exploded.groupby(df_exploded.index)['country'].apply(
    lambda x: x.unique().tolist()
)
df["country"] = cleaned_country_series

#### Remove old cols and fill some NA

In [ ]:
df.drop('duration', axis=1, inplace=True)
df.drop("rating_clean", axis=1, inplace=True)
df.drop("listed_in", axis=1, inplace=True)

In [ ]:
df['duration_class'] = df['duration_class'].apply(lambda x: x if x is None else 'Unknown')
df['genre'] = df['genre'].apply(lambda x: x if isinstance(x, list) else ['Unknown'])

In [ ]:
df.info()

### Content Distribution Analysis

#### Movies vs Tv Show

In [ ]:
type_counts = df['type'].value_counts()
type_percent = df['type'].value_counts(normalize=True) * 100

plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='type')
plt.title('Movies vs TV Shows')
plt.show()

In [ ]:
if len(df['streaming'].unique()) > 1:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='streaming', hue='type')
    plt.title('Content by streaming and type')
    plt.xticks(rotation=45)
    plt.show()

#### Countries

In [ ]:
N = 10

top_n_countries = df.explode('country')['country'].value_counts().head(N)

plt.figure(figsize=(12, 7))
top_n_countries.plot(kind='bar', color='skyblue')

plt.title(f'Top {N} Countries Most Common')
plt.xlabel('Contry')
plt.ylabel('Titles number')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
N = 10
streaming_services = df['streaming'].unique()

for service in streaming_services:
    
    df_service = df[df['streaming'] == service]
    
    top_n_countries = df_service.explode('country').dropna(subset=['country'])['country'].value_counts().head(N)
    
    plt.figure(figsize=(12, 7))
    top_n_countries.plot(kind='bar', color='skyblue')
    
    plt.title(f'Top {N} Countries Most Common in {service}')
    plt.xlabel('Country')
    plt.ylabel('Titles Number')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

#### Genres

In [ ]:
N = 10

top_n_genres = df.explode('genre')['genre'].value_counts().head(N)

plt.figure(figsize=(12, 7))
top_n_genres.plot(kind='bar', color='skyblue')

plt.title(f'Top {N} Genres Most Common')
plt.xlabel('Genre')
plt.ylabel('Titles number')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
N = 10
streaming_services = df['streaming'].unique()

for service in streaming_services:
    
    df_service = df[df['streaming'] == service]
    
    top_n_countries = df_service.explode('genre').dropna(subset=['genre'])['genre'].value_counts().head(N)
    
    plt.figure(figsize=(12, 7))
    top_n_countries.plot(kind='bar', color='skyblue')
    
    plt.title(f'Top {N} Genres Most Common in {service}')
    plt.xlabel('Genre')
    plt.ylabel('Titles Number')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

#### Time

In [ ]:
plt.figure(figsize=(14, 10))

df['year_added'] = pd.to_datetime(df['date_added']).dt.year

content_added_by_year = df.groupby(['year_added', 'streaming']).size().unstack(fill_value=0)

plt.subplot(2, 2, 1)
content_added_by_year.plot(kind='line', marker='o', ax=plt.gca())
plt.title('Content Added by year and plataform=')
plt.ylabel('Content Count')
plt.xlabel('Release Year')
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
release_year_counts = df['release_year'].value_counts().sort_index()
plt.plot(release_year_counts.index, release_year_counts.values)
plt.title('Distribution by release year')
plt.xlabel('Release Year')
plt.ylabel('Content Count')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

df['content_era'] = pd.cut(df['release_year'], 
                          bins=[1900, 2000, 2015, 2022],
                          labels=['Classic (pre-2000)', 'Modern (2000-2015)', 'Recent (2016+)'])

era_by_platform = pd.crosstab(df['streaming'], df['content_era'], normalize='index') * 100
era_by_platform.plot(kind='bar', stacked=True, ax=plt.gca())
plt.title('Content Era Distribution by Streaming Platform')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=45)
plt.legend(title='Content Era', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
df['age_when_added'] = df['year_added'] - df['release_year']
age_stats = df.groupby('streaming')['age_when_added'].agg(['mean', 'median'])
age_stats.plot(kind='bar', ax=plt.gca())
plt.title('Content Age When Added to Platform')
plt.ylabel('Years')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

#### Rating

In [ ]:
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
rating_by_streaming = pd.crosstab(df['streaming'], df['rating'])
rating_by_streaming.plot(kind='bar', stacked=True, ax=plt.gca())
plt.title('Ratings Distribution by plataform')
plt.ylabel('Content Count')
plt.xlabel('Streaming Service')
plt.xticks(rotation=45)
plt.legend(title='Rating', bbox_to_anchor=(1.05, 1), loc='upper left')